# 04. Team Persistence

Notebooks 01–03 introduced the moving parts of an Arc team: formation (`TeamConfig`, `EntityRegistry`), task distribution (`Channel`, `Priority`, `MsgType`), and inter-agent messaging (`MessagingService`, `Cursor`). Everything in those notebooks ran cleanly in process.

This notebook is about what happens when the process stops.

When an agent crashes mid-task, when an audit chain has to survive an OS reboot, when tomorrow's investigator needs to read today's messages — the team has to remember. That memory has to be tamper-evident, recoverable, and isolated per-agent. ArcTeam ships a **storage abstraction** (`StorageBackend` Protocol with `MemoryBackend` for tests and `NatsBackend` — NATS JetStream — for production), a **shared file store** (`TeamFileStore`), a **knowledge graph facade** (`TeamMemoryService` over `TeamMemoryConfig`), and a **chained, Ed25519-signed audit log** (`AuditLogger` with `verify_chain`). All four collaborate on one job: *the team survives a restart with its history intact*.

You will learn:

- Why persistence is a federal-tier requirement, not a nice-to-have
- The `StorageBackend` Protocol and how `MemoryBackend` (tests) and `NatsBackend` (production, JetStream-backed) implement it
- How `NatsBackend` maps records to JetStream KV buckets and streams to JetStream streams — durability and durable-consumer resume come from JetStream itself, not from custom file I/O
- How `TeamFileStore` isolates per-agent shared workspaces and rejects path traversal
- How `TeamMemoryService` + `TeamMemoryConfig` manage the team knowledge graph as a Null Object when disabled
- How `AuditLogger` chains a per-record Ed25519 (or ECDSA-P256) signature over the previous record's signature, and verifies the chain after restart
- The recovery flow: cold backend → load last seq/signature → append → verify
- Operational guidance: backups, durable JetStream retention, file permissions, signer/key custody

Cross-references:

- `01-team-formation.ipynb` introduced `TeamConfig` and the `EntityRegistry`. We assume that surface.
- `02-task-distribution.ipynb` covered `Channel`, `MsgType`, `Priority`. We won't repeat the routing demo.
- `03-messaging-channels.ipynb` walked `MessagingService` end-to-end. We use audit alongside it but won't re-derive the polling protocol.

This notebook is **mock-first**: every executed cell runs without a network connection, using `tmp_path`-style temp directories and `MemoryBackend`. `NatsBackend` requires a running NATS/JetStream server — we cover its real persistence model and exercise its pure, connection-free helpers, but do not connect to a live server (that infra is out of scope for this walkthrough).

## 1. Setup

Standard Arc walkthrough preamble: locate the repo root and add every package's `src/` to `sys.path`. No API keys are needed — every example below is mock-first.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Smoke-import the public persistence surface. If any of these fail, the rest of the notebook will not run — fix `sys.path` first.

In [2]:
import hashlib

import arcteam
from arctrust.signer import ED25519, InProcessSigner
from arcteam import (
    AuditLogger,
    AuditRecord,
    MemoryBackend,
    NatsBackend,
    StorageBackend,
    TeamConfig,
    TeamFileStore,
    TeamMemoryConfig,
    TeamMemoryService,
)
from arcteam.memory.types import (
    Classification,
    EntityMetadata,
)

print("arcteam version:", arcteam.__version__)
print("Public persistence surface imported OK.")

arcteam version: 0.5.0
Public persistence surface imported OK.


We will use a tiny helper that returns a fresh temp directory we own. In real test code, pytest's `tmp_path` fixture does this for you; in a notebook we have to allocate one explicitly so each cell that needs disk-backed storage starts clean.

In [3]:
import tempfile


def fresh_tmp(label: str = "") -> Path:
    """Return a brand-new temp directory. The OS reaps it on shutdown."""
    return Path(tempfile.mkdtemp(prefix=f"arcteam-walkthrough-{label}-"))


DEMO_ROOT = fresh_tmp("demo")
print("Demo root:", DEMO_ROOT)
print("Exists:", DEMO_ROOT.exists(), "  Empty:", not any(DEMO_ROOT.iterdir()))

Demo root: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-demo-56z9vmv7
Exists: True   Empty: True


## 2. Why persistence — survival across restarts, audit, recovery

Three real-world failure modes drive ArcTeam's persistence design:

1. **Process crash mid-poll.** An agent receives a message, starts working, then segfaults before acknowledging. On restart it must see *the same message again* — the cursor must persist durably, not in memory.
2. **Audit reconstruction after the fact.** A compliance officer six months from now needs to answer "who told the procurement agent to dispatch task X, and at what time?" The log must be append-only, ordered, and tamper-evident.
3. **Shared workspace handoff.** Agent A produces a CSV; agent B picks it up an hour later. Without a shared on-disk surface they have to re-do A's work.

ArcTeam's answer to all three is the same shape: **a swappable backend behind a `Protocol`.** In tests you use `MemoryBackend` for speed; in production you use `NatsBackend`, which persists the same records and streams via NATS JetStream — durable KV buckets for records, durable JetStream streams (with server-assigned publish sequence and durable consumers) for append-only logs. The `StorageBackend` Protocol is the same either way, so messenger/registry/audit code never branches on which backend is wired in.

The four building blocks we cover here:

| Component | Persistence concern | What it stores |
|---|---|---|
| `StorageBackend` (Protocol) | Abstract interface | — |
| `MemoryBackend` | In-process, ephemeral | Dict-of-dicts (tests) |
| `NatsBackend` | JetStream-backed, durable | KV buckets (records), JetStream streams (append logs) — production |
| `TeamFileStore` | Shared workspaces between agents | Files (any format) |
| `TeamMemoryService` + `TeamMemoryConfig` | Knowledge graph, decisions, search | Markdown entities + JSONL decisions |
| `AuditLogger` | Tamper-evident chain of events | Ed25519-signed, hash-chained records |

The thread connecting them: every collection of "things that change over time" is an append-only stream, every singleton record is a keyed write, and every tamper-sensitive operation is signed forward into the next record via a chained signature. That's it.

In [4]:
# A quick sketch of the three failure modes — we'll demonstrate them concretely below.
failure_modes = [
    ("Crash mid-poll", "Cursor must survive process death", "NatsBackend record (JetStream KV)"),
    ("Audit reconstruction", "Chain must be tamper-evident", "AuditLogger.verify_chain()"),
    ("Shared workspace handoff", "Files exchanged out-of-band", "TeamFileStore"),
]
for name, requirement, mechanism in failure_modes:
    print(f"  {name:30s}  →  {requirement:40s}  via  {mechanism}")

  Crash mid-poll                  →  Cursor must survive process death         via  NatsBackend record (JetStream KV)
  Audit reconstruction            →  Chain must be tamper-evident              via  AuditLogger.verify_chain()
  Shared workspace handoff        →  Files exchanged out-of-band               via  TeamFileStore


## 3. `StorageBackend` Protocol + `MemoryBackend` (in-process)

`StorageBackend` is a `runtime_checkable` `Protocol` defined in `arcteam.storage`. It declares the entire surface every persistence component depends on:

- **Records** (singleton JSON): `read`, `write`, `delete`, `exists`, `query`, `list_keys`
- **Streams** (append-only): `append`, `append_auto_seq`, `read_stream`, `read_last`, `get_stream_end_byte_pos`
- **Durable consumers**: `open_consumer` — a fetch/ack handle that resumes from its last-acked position on rebind (REQ-021)

Two implementations ship in the box. `MemoryBackend` keeps everything in dicts — ideal for unit tests because it's instantaneous and process-local; `packages/arcteam/tests/unit/test_storage.py` exercises it directly. `NatsBackend` (covered in section 4) implements the identical Protocol over NATS JetStream — the production substrate. There is no file-based backend in ArcTeam; on-disk durability for records/streams comes from JetStream, not from a custom JSON/JSONL writer.

In [5]:
# StorageBackend is a runtime_checkable Protocol — MemoryBackend satisfies it directly.
mem = MemoryBackend()
print("MemoryBackend is a StorageBackend:", isinstance(mem, StorageBackend))

# NatsBackend needs a live JetStreamContext to construct (NatsBackend.connect(...)),
# so we can't instantiate it here — but we can show, purely offline, that it
# implements the same method surface the Protocol requires.
protocol_methods = {
    name for name in dir(StorageBackend) if not name.startswith("_")
}
nats_methods = {name for name in dir(NatsBackend) if not name.startswith("_")}
print("NatsBackend implements every StorageBackend method:", protocol_methods <= nats_methods)
print("Same surface — code that takes a StorageBackend works with either.")

MemoryBackend is a StorageBackend: True
NatsBackend implements every StorageBackend method: True
Same surface — code that takes a StorageBackend works with either.


Records first. `write` then `read` gives you back exactly what you wrote — everything goes through the backend's own JSON-compatible encoding, so the data must be JSON-serializable. `NatsBackend` implements the identical calls against a JetStream KV bucket per collection (section 4).

In [6]:
import asyncio


async def demo_record_basics(backend: StorageBackend) -> None:
    await backend.write(
        "entities",
        "agent_alice",
        {"id": "agent_alice", "role": "analyst"},
    )
    print(" exists:", await backend.exists("entities", "agent_alice"))
    print(" read  :", await backend.read("entities", "agent_alice"))
    print(" miss  :", await backend.read("entities", "agent_unknown"))


print("MemoryBackend:")
await demo_record_basics(mem)

MemoryBackend:
 exists: True
 read  : {'id': 'agent_alice', 'role': 'analyst'}
 miss  : None


Streams are append-only. `append_auto_seq` is the safer of the two append paths — it assigns a monotonic `seq` under a lock (in `MemoryBackend`) or lets JetStream's own publish sequence do the same job (in `NatsBackend`) and returns it back. That's how the messaging layer guarantees no two messages collide on the same sequence number even across processes.

In [7]:
async def demo_stream_basics(backend: StorageBackend) -> None:
    for body in ["hello", "world", "again"]:
        seq, offset = await backend.append_auto_seq(
            "messages",
            "channel.ops",
            {"body": body},
        )
        print(f"  appended seq={seq:2d} at byte_pos={offset}")
    last = await backend.read_last("messages", "channel.ops")
    print("  last entry:", last)
    end_pos = await backend.get_stream_end_byte_pos("messages", "channel.ops")
    print("  end byte position:", end_pos)


print("MemoryBackend stream:")
await demo_stream_basics(MemoryBackend())

MemoryBackend stream:
  appended seq= 1 at byte_pos=0
  appended seq= 2 at byte_pos=28
  appended seq= 3 at byte_pos=56
  last entry: {'body': 'again', 'seq': 3}
  end byte position: 84


Read patterns. `read_stream` accepts `after_seq` and `byte_pos` for resumption — the messaging layer's `Cursor` records both values so a reader can continue exactly where it left off without re-scanning the whole file.

In [8]:
stream_demo = MemoryBackend()
for i in range(1, 6):
    await stream_demo.append_auto_seq("messages", "ops", {"body": f"m{i}"})

all_msgs = await stream_demo.read_stream("messages", "ops", after_seq=0)
print("All msgs:", [m["body"] for m in all_msgs])

after_two = await stream_demo.read_stream("messages", "ops", after_seq=2)
print("After seq=2:", [m["body"] for m in after_two])

first_three = await stream_demo.read_stream("messages", "ops", after_seq=0, limit=3)
print("First three with limit=3:", [m["body"] for m in first_three])

All msgs: ['m1', 'm2', 'm3', 'm4', 'm5']
After seq=2: ['m3', 'm4', 'm5']
First three with limit=3: ['m1', 'm2', 'm3']


Queries do field filtering and prefix filtering on records. They're scoped per-collection; `MemoryBackend` walks its dict, `FileBackend` globs the directory.

In [9]:
qb = MemoryBackend()
await qb.write("entities", "agent_a", {"id": "a", "type": "agent", "role": "ops"})
await qb.write("entities", "agent_b", {"id": "b", "type": "agent", "role": "dev"})
await qb.write("entities", "user_c", {"id": "c", "type": "user", "role": "ops"})

by_type = await qb.query("entities", filters={"type": "agent"})
print("agents:", [e["id"] for e in by_type])

by_role = await qb.query("entities", filters={"role": "ops"})
print("ops:   ", [e["id"] for e in by_role])

agents_only = await qb.query("entities", prefix="agent_")
print("prefix:", [e["id"] for e in agents_only])

all_keys = await qb.list_keys("entities")
print("keys:  ", all_keys)

agents: ['a', 'b']
ops:    ['a', 'c']
prefix: ['a', 'b']
keys:   ['agent_a', 'agent_b', 'user_c']


## 4. `NatsBackend` — production persistence over JetStream

`NatsBackend` is the real durability layer: it implements `StorageBackend` over NATS JetStream so the messenger, registry, and audit code above run *unchanged* whether they're pointed at an in-process dict or a durable, replicated JetStream deployment.

- **Records → a JetStream KV bucket per collection.** `read`/`write`/`delete`/`query`/`list_keys`/`exists` map onto `kv.get`/`kv.put`/`kv.delete` against a bucket named `b{hex(collection)}`.
- **Streams → a JetStream stream per `(collection, key)`.** `append`/`append_auto_seq`/`read_stream`/`read_last` map onto a JetStream stream named `s{hex(collection + "\x00" + key)}`. The **JetStream publish sequence is the authoritative message `seq`** — there is no client-side counter to keep in sync; the broker assigns it.
- **Durable consumers → `open_consumer`.** Re-opening the same `durable` name resumes exactly where the last-acked message left off — this is what gives an agent live push *and* resume-on-restart (REQ-021) with no extra bookkeeping.
- **Names are hex-encoded, not path-sanitized.** Collection/key pairs frequently aren't valid NATS subject tokens (they can contain `://`, `.`, arbitrary Unicode). Rather than validating and rejecting unsafe characters (a filesystem's `_sanitize`-style problem), `NatsBackend` just encodes anything that isn't already a safe subject into a reversible hex token. There is no "escape the root" failure mode because there is no root — it's not a filesystem.

Three properties fall out of using JetStream rather than a hand-rolled file store:

- **Durability** is the broker's job — replicated, fsync'd storage per JetStream's own configuration. No client-side atomic-rename trick needed.
- **Ordering/atomicity** is enforced by the server: every publish to a given subject gets a strictly increasing sequence number assigned by JetStream itself, so concurrent writers *across hosts* — not just across threads in one process — never collide on `seq`. No `fcntl.flock` equivalent is needed client-side.
- **Crash-safety** is per-message: JetStream persists each published message as a whole unit, so there is no "torn line" failure mode analogous to a half-written JSONL append — a crash either lands a message or doesn't.

Constructing a real one takes a live server (`await NatsBackend.connect(servers)`), which this notebook doesn't have. What follows are the pure, connection-free parts of `NatsBackend` — the naming scheme — which we exercise directly, plus the constructor's real signature via introspection.

In [10]:
# The naming helpers are pure functions — no NATS connection required to exercise them.
import inspect

from arcteam.backends.nats import _bucket_name, _hex, _stream_name, _subject, _unhex

print("bucket for 'entities' collection   :", _bucket_name("entities"))
print("stream for messenger stream key    :", _stream_name("messages", "arc.channel.alpha"))

# arc.channel.alpha is already a safe NATS subject — passes through unchanged.
print("subject for a safe key             :", _subject("arc.channel.alpha"))
# A key with characters NATS subjects don't allow falls back to a reversible hex token.
print("subject for an unsafe key          :", _subject("agent://alice#dm"))

# Round-trip: hex-encoding is reversible, so a key is always recoverable.
token = _hex("agent://alice#dm")
print("hex round-trip                     :", _unhex(token) == "agent://alice#dm")

print("\nNatsBackend.connect signature      :", inspect.signature(NatsBackend.connect))

bucket for 'entities' collection   : b656e746974696573
stream for messenger stream key    : s6d65737361676573006172632e6368616e6e656c2e616c706861
subject for a safe key             : arc.channel.alpha
subject for an unsafe key          : k6167656e743a2f2f616c69636523646d
hex round-trip                     : True

NatsBackend.connect signature      : (servers: 'str | list[str]', *, connect_timeout: 'float' = 3.0) -> 'NatsBackend'


**Concurrent writers get unique, ordered `seq` numbers — for different reasons per backend.** In production, `NatsBackend` relies on JetStream: every publish to a stream's subject is sequenced by the broker itself, so concurrent writers *from different hosts* never collide. In `MemoryBackend`, `append_auto_seq` has no internal lock at all — it doesn't need one, because it contains no `await` point between reading the last `seq` and appending the new entry, so Python's cooperative single-threaded event loop never interleaves two calls mid-function. That's a genuine (if easy to miss) correctness property worth demonstrating directly, since it's the reason `MemoryBackend` is safe to use in concurrent test code without its own locking.

In [11]:
concur = MemoryBackend()


async def writer(idx: int) -> int:
    seq, _ = await concur.append_auto_seq(
        "streams",
        "shared",
        {"writer": idx},
    )
    return seq


seqs = await asyncio.gather(*(writer(i) for i in range(10)))
print("unique seq values:", sorted(seqs) == list(range(1, 11)))
print("  observed       :", sorted(seqs))

unique seq values: True
  observed       : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 5. `TeamFileStore` — workspace files between agents

Sometimes agents need to exchange artifacts that aren't a fit for messages: large CSVs, generated reports, intermediate caches. `TeamFileStore` carves out a per-agent subdirectory under `<team_root>/shared/files/<agent_name>/` and copies files into it.

Two guarantees:

- **Path-within validation.** Every destination is resolved and checked against the team root. A malicious or buggy `agent_name` cannot escape the shared tree.
- **No silent overwrites.** If a destination filename already exists, a numeric suffix is appended (`report.pdf` → `report_1.pdf`).

In [12]:
team_root = fresh_tmp("team")
store = TeamFileStore(team_root=team_root)
print("shared dir:", store.shared_dir)
print("exists yet?", store.shared_dir.exists())  # Created lazily on first store()

shared dir: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-team-ql81eitd/shared/files
exists yet? False


Stage a fake artifact and copy it in. The returned dict has the destination path, the (possibly suffixed) filename, the size, and the agent that owns it.

In [13]:
staging = fresh_tmp("staging")
src = staging / "report.csv"
src.write_text("col_a,col_b\n1,2\n3,4\n")

info = await store.store(source_path=src, agent_name="alice")
print(info)

# Store the same name twice — second copy gets a suffix.
info2 = await store.store(source_path=src, agent_name="alice")
print(info2)

{'path': '/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-team-ql81eitd/shared/files/alice/report.csv', 'filename': 'report.csv', 'size': 20, 'agent': 'alice'}
{'path': '/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-team-ql81eitd/shared/files/alice/report_1.csv', 'filename': 'report_1.csv', 'size': 20, 'agent': 'alice'}


List the shared workspace. Pass an `agent_name` to filter; omit it to list every agent.

In [14]:
# Stage another file for a different agent so the cross-agent listing has more than one row.
src_b = staging / "summary.txt"
src_b.write_text("all checks green\n")
await store.store(source_path=src_b, agent_name="bob")

alice_files = await store.list_files(agent_name="alice")
all_files = await store.list_files()

print("alice has:")
for f in alice_files:
    print("  ", f["filename"], "-", f["size"], "bytes")

print("\nteam-wide:")
for f in all_files:
    print(f"  [{f['agent']}] {f['filename']} ({f['size']} bytes)")

alice has:
   report.csv - 20 bytes
   report_1.csv - 20 bytes

team-wide:
  [alice] report.csv (20 bytes)
  [alice] report_1.csv (20 bytes)
  [bob] summary.txt (17 bytes)


Path traversal rejection. An `agent_name` containing `..` or other unsafe characters is refused before any filesystem operation runs.

In [15]:
for bad_name in ["../escape", "alice/../../etc", "with spaces", "weird;chars"]:
    try:
        await store.store(source_path=src, agent_name=bad_name)
        print(f"  {bad_name!r:30s} -> ALLOWED (bug!)")
    except ValueError:
        print(f"  {bad_name!r:30s} -> rejected")
    except FileNotFoundError:
        # If a name is borderline-valid the store call may still fail later;
        # we only care that path-escape names are caught at the validator.
        print(f"  {bad_name!r:30s} -> not blocked at validator (would fail later)")

# A missing source file is its own clean error (FileNotFoundError, not ValueError).
try:
    await store.store(source_path=staging / "does_not_exist.txt", agent_name="alice")
except FileNotFoundError as exc:
    print("\n  missing source -> FileNotFoundError:", exc)

  '../escape'                    -> rejected
  'alice/../../etc'              -> rejected
  'with spaces'                  -> rejected
  'weird;chars'                  -> rejected

  missing source -> FileNotFoundError: Source file not found: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-staging-3_xqb7u4/does_not_exist.txt


## 6. `TeamMemoryService` + `TeamMemoryConfig`

`TeamMemoryService` is the public facade for the team knowledge graph. It coordinates an `IndexManager`, a `SearchEngine`, a `PromotionGate`, a `ClassificationChecker`, and a `MemoryStorage` — all wired up on construction from a single `TeamMemoryConfig`. We don't deep-dive each component here (that's a topic in its own right). What we cover is the **persistence-relevant surface**:

- The Null Object pattern when `enabled=False`
- How `record_decision` appends to a JSONL file under `<root>/decisions.jsonl`
- The `entities_dir` and `index_path` derived from `root`
- The `status()` snapshot used by health checks

In [16]:
memory_root = fresh_tmp("memory")
cfg = TeamMemoryConfig(
    enabled=True,
    root=memory_root,
)
print("enabled       :", cfg.enabled)
print("root          :", cfg.root)
print("entities_dir  :", cfg.entities_dir)
print("index_path    :", cfg.index_path)
print("entity_types  :", cfg.entity_types)
print("per_entity_budget tokens:", cfg.per_entity_budget)
print("tier          :", cfg.tier)

enabled       : True
root          : /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-memory-_9rjobsg
entities_dir  : /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-memory-_9rjobsg/entities
index_path    : /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-memory-_9rjobsg/entities/_index.json
entity_types  : ['person', 'organization', 'project', 'domain', 'process', 'playbook']
per_entity_budget tokens: 800
tier          : personal


Construct the service. `TeamMemoryService` accepts an optional `audit_logger` — when provided, every `search` call emits a `memory.search` audit event. We pass `None` here for clarity.

In [17]:
svc = TeamMemoryService(config=cfg, audit_logger=None)

status = await svc.status()
print("status         :", status)

status         : enabled=True entity_count=0 index_dirty=False last_consolidated='' entities_path=''


`record_decision` is the simplest persistence path through the memory service. It serializes a dict to JSONL under `<root>/decisions.jsonl` with a UTC timestamp and the recording agent's id. After a restart you can `tail` this file to reconstruct the decision history.

In [18]:
import json

await svc.record_decision(
    decision={
        "title": "Adopt NatsBackend for shared storage",
        "rationale": "Survives process restart; JetStream sequencing keeps concurrent writers safe.",
        "alternatives": ["In-memory dict", "SQLite"],
    },
    agent_id="agent://alice",
)
await svc.record_decision(
    decision={"title": "Rotate the audit signer quarterly", "owner": "agent://bob"},
    agent_id="agent://bob",
)

decisions_log = cfg.root / "decisions.jsonl"
print("decisions log:", decisions_log)
for line in decisions_log.read_text().splitlines():
    parsed = json.loads(line)
    print(f"  [{parsed['timestamp']}] {parsed['agent_id']}  -  {parsed.get('title', '')}")

decisions log: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcteam-walkthrough-memory-_9rjobsg/decisions.jsonl
  [2026-07-09T12:23:08.913710+00:00] agent://alice  -  Adopt NatsBackend for shared storage
  [2026-07-09T12:23:08.913984+00:00] agent://bob  -  Rotate the audit signer quarterly


List entities exercises the index. With a fresh root it returns an empty list — no entities have been promoted yet. `Classification.UNCLASSIFIED` is the lowest clearance tier and matches every entity that hasn't been marked as restricted.

In [19]:
entities = await svc.list_entities(agent_classification=Classification.UNCLASSIFIED)
print("entities (empty graph):", entities)

results = await svc.search(
    query="persistence",
    agent_classification=Classification.UNCLASSIFIED,
    max_results=5,
    agent_id="agent://alice",
)
print("search results       :", results)

entities (empty graph): []
search results       : []


Null Object pattern. Disable the service via config; every method still answers — `search` returns `[]`, `promote` returns a `PromotionResult` flagged `disabled`, and `status` reports `enabled=False`. Calling code does not need to know whether memory is on.

In [20]:
off_cfg = TeamMemoryConfig(enabled=False, root=fresh_tmp("memory-off"))
off_svc = TeamMemoryService(config=off_cfg)

print("status:", await off_svc.status())
print("search:", await off_svc.search("anything"))
print("list  :", await off_svc.list_entities())
print("get   :", await off_svc.get_entity("some_id"))

promotion = await off_svc.promote(
    entity_id="alice",
    content="# Alice\n\nAnalyst.",
    metadata=EntityMetadata(
        entity_type="person",
        entity_id="alice",
        name="Alice",
    ),
    agent_id="agent://alice",
)
print("promote:", promotion)

status: enabled=False entity_count=0 index_dirty=False last_consolidated='' entities_path=''
search: []
list  : []
get   : None
promote: success=False entity_id='alice' action='disabled' message='Team memory is disabled'


## 7. `AuditLogger` — chain integrity, `verify_chain`

`AuditLogger` is ArcTeam's tamper-evident log. Every record carries an `audit_seq` (monotonic counter), the operational metadata (`event_type`, `subject`, `actor_id`, `target_id`, `classification`, `detail`, `timestamp_utc`), and a per-record **asymmetric signature** (`signature`/`public_key`/`algorithm`) that *chains*: each signature is computed over `prev_signature || canonical(record)`, so mutating any record breaks every subsequent signature. This is deliberately **not** an HMAC — an HMAC verifier holds the same secret needed to forge a record, which defeats non-repudiation (AU-10). A verifier here only ever needs the signer's *public* key.

Three operational notes:

1. **Provide a `Signer`.** `AuditLogger(backend, signer)` takes an `arctrust.signer.Signer` — typically `InProcessSigner(seed, ED25519)` for personal/enterprise custody, or a `VaultSigner`/`build_signer(SignerConfig(custody="vault_transit", ...))` at federal tier so the seed never enters this process (SPEC-037 REQ-006). This is the same `Signer` seam used by `arctrust.audit.WormSink` (see `arctrust/04-audit-sinks.ipynb`).
2. **Always `await initialize()`.** It loads the last `audit_seq` and `prev_signature` from the backend so a restart picks up where the previous process left off.
3. **`verify_chain` is batched.** It walks the stream `_VERIFY_BATCH_SIZE` records at a time, so verifying a chain of millions of entries does not blow up memory. Verification checks against the signer's *known* public key, never a key embedded in a record — so a record re-signed under a substituted key still fails.

In [21]:
audit_backend = MemoryBackend()
audit = AuditLogger(
    audit_backend,
    signer=InProcessSigner(hashlib.sha256(b"secret-key-32-bytes-min---------").digest(), ED25519),
)
await audit.initialize()
print("initial seq    :", audit._seq)
print("prev_signature :", audit._prev_signature or "(empty — first record)")

initial seq    : 0
prev_signature : (empty — first record)


Log a few events. Each `log()` call assigns the next `audit_seq`, computes the signature over the prior signature + this record's canonical bytes, and appends. The model fields come from `arcteam.types.AuditRecord`.

In [22]:
events = [
    ("team.formed", "team-alpha", "created with 3 members"),
    ("message.sent", "arc.channel.ops", "task assigned"),
    ("message.delivered", "arc.channel.ops", "agent://bob received"),
    ("task.completed", "task-42", "agent://bob done"),
]
for event_type, subject, detail in events:
    await audit.log(
        event_type=event_type,
        subject=subject,
        actor_id="agent://alice",
        detail=detail,
        target_id="agent://bob",
    )

records = await audit_backend.read_stream("audit", "audit", after_seq=0)
for r in records:
    print(f"  seq={r['audit_seq']}  {r['event_type']:20s}  signature={r['signature'][:12]}…")

  seq=1  team.formed           signature=f3f21a13d0dc…
  seq=2  message.sent          signature=5c273a37a0d1…
  seq=3  message.delivered     signature=1122ad59f054…
  seq=4  task.completed        signature=e32899cc8716…


Verify the chain. With no tampering, every signature re-verifies against the stored value and `verify_chain` returns `(True, last_seq)`.

In [23]:
valid, last_verified = await audit.verify_chain()
print("chain valid       :", valid)
print("last verified seq :", last_verified)

chain valid       : True
last verified seq : 4


Tamper test. We reach into the in-memory backend and mutate a record's `detail`. The next `verify_chain` call returns `False` along with the last seq that was still valid — investigators get an exact pointer to the breach.

In [24]:
stored = audit_backend._streams["audit"]["audit"]
print("before tamper, record 2 detail:", stored[1]["detail"])
stored[1]["detail"] = "FORGED"

valid, last_verified = await audit.verify_chain()
print("chain valid       :", valid)
print("last good seq     :", last_verified, " (everything after this is suspect)")

Audit signature mismatch at seq 2


before tamper, record 2 detail: task assigned
chain valid       : False
last good seq     : 1  (everything after this is suspect)


Loading the signer from the environment. There's no `AuditLogger.load_key()` helper — key/signer construction is the caller's job, via `arctrust.signer`. The common pattern is `build_signer(SignerConfig(...), seed=...)`: `custody="in_process"` resolves an `InProcessSigner` from a seed (e.g. pulled from an env var backed by your secret store at boot); `custody="vault_transit"` resolves a `VaultSigner` that signs by reference through a Vault Transit / HSM / KMS boundary — the seed then never enters this process at all (SPEC-037 REQ-006), which is the federal-tier posture.

In [25]:
# Simulate the env being set, then resolve an in-process signer from it.
from arctrust.signer import IN_PROCESS, SignerConfig, build_signer

os.environ["ARCTEAM_SIGNING_SEED"] = "persistent-seed-from-secret-store-32b!!"
seed_from_env = hashlib.sha256(os.environ["ARCTEAM_SIGNING_SEED"].encode()).digest()

env_signer = build_signer(SignerConfig(custody=IN_PROCESS, algorithm=ED25519), seed=seed_from_env)
print("resolved signer public key:", env_signer.public_key.hex()[:32], "...")
del os.environ["ARCTEAM_SIGNING_SEED"]

# in_process custody with no seed is a hard error — fail closed (NFR-3), never
# a silent downgrade.
try:
    build_signer(SignerConfig(custody=IN_PROCESS, algorithm=ED25519))
except Exception as exc:
    print(f"no seed -> {type(exc).__name__}: {exc}")

resolved signer public key: b539022d2312172b8cac3bae9bc0fb38 ...
no seed -> SignerError: in_process custody requires a seed


## 8. Recovery — load state from disk, replay, sanity checks

The whole point of `NatsBackend` + `AuditLogger.initialize()` is that a fresh process can come up against an existing durable JetStream state and continue without losing or duplicating anything. This notebook has no live NATS server, so we can't literally kill a Python process and restart it against the same on-disk store the way we could with `NatsBackend`. What we *can* demonstrate faithfully, without any infra, is the exact **recovery mechanism** `AuditLogger` runs on restart — because that mechanism only depends on the `StorageBackend` Protocol, not on which implementation backs it:

1. Process A runs, writes records and audit events, into a backend that represents the durable store.
2. Process A's *service objects* (`AuditLogger`, etc.) are discarded — but the durable store itself survives (in production: JetStream; here: the same `MemoryBackend` instance kept alive, standing in for "the durable substrate").
3. Process B constructs *fresh* service objects against that same durable store and *recovers* the prior state via `await audit.initialize()`.

The `initialize()` call is the load-bearing step — it reads `read_last` off the audit stream and pulls the prior `audit_seq` and `prev_signature` forward. That call is identical whether the backend underneath is `MemoryBackend` or `NatsBackend`.

In [26]:
# --- Process A: warm setup ---
# recovery_store stands in for the durable substrate (NatsBackend/JetStream in
# production). We keep this object alive across the "restart" below and only
# discard process A's service objects — exactly the boundary that matters.
recovery_store = MemoryBackend()
RECOVERY_SEED = hashlib.sha256(b"persistent-test-key-32bytes-----").digest()

audit_a = AuditLogger(recovery_store, signer=InProcessSigner(RECOVERY_SEED, ED25519))
await audit_a.initialize()

# Write a few records and a few audit events.
await recovery_store.write("entities", "alice", {"id": "alice", "role": "analyst"})
await recovery_store.write("entities", "bob", {"id": "bob", "role": "engineer"})
for i in range(3):
    await recovery_store.append_auto_seq("messages", "channel.ops", {"body": f"task-{i}"})
for i, event in enumerate(["team.formed", "member.joined", "task.dispatched"], start=1):
    await audit_a.log(
        event_type=event,
        subject=f"subject-{i}",
        actor_id="agent://alice",
        detail="warm-up",
    )

print("process A audit_seq   :", audit_a._seq)
print("entities recorded     :", await recovery_store.list_keys("entities"))
print("channel.ops stream len:", len(await recovery_store.read_stream("messages", "channel.ops", after_seq=0)))

process A audit_seq   : 3
entities recorded     : ['alice', 'bob']
channel.ops stream len: 3


Simulate the process restart. Drop process A's *service object* reference — `recovery_store` (the durable substrate) stays alive, exactly like a JetStream deployment would survive a process exit.

In [27]:
# --- Pretend process A crashed: drop its service object. ---
del audit_a
print("process A's AuditLogger discarded — only recovery_store (the durable substrate) survives.")

process A's AuditLogger discarded — only recovery_store (the durable substrate) survives.


Process B starts cold. It constructs a fresh `AuditLogger` against `recovery_store` — the same durable substrate, with the same signing key material (in production, resolved from the same vault/secret-store reference process A used). The crucial step is `await audit.initialize()` — that reads `read_last` from the audit stream and pulls the prior `audit_seq` and `prev_signature` forward. Against a real `NatsBackend`, "process B" would instead reconnect (`await NatsBackend.connect(servers)`) to the same JetStream deployment; the `AuditLogger` code path is unchanged either way.

In [28]:
# --- Process B: cold start, same durable store, same signing key ---
audit_b = AuditLogger(recovery_store, signer=InProcessSigner(RECOVERY_SEED, ED25519))
await audit_b.initialize()

print("process B recovered audit_seq   :", audit_b._seq)
print("process B recovered prev_signature:", audit_b._prev_signature[:16], "…")

# Records and streams are visible without any extra work — they live in the store.
alice = await recovery_store.read("entities", "alice")
msgs = await recovery_store.read_stream("messages", "channel.ops", after_seq=0)
print("recovered alice :", alice)
print("recovered msgs  :", [m["body"] for m in msgs])

process B recovered audit_seq   : 3
process B recovered prev_signature: 214c8be3581c1b0b …
recovered alice : {'id': 'alice', 'role': 'analyst'}
recovered msgs  : ['task-0', 'task-1', 'task-2']


Now the new process appends its own audit events. The chain continues — the new signature is computed off the *prior* `prev_signature` loaded from the store, so verification covers both pre-restart and post-restart events as one continuous chain.

In [29]:
await audit_b.log(
    event_type="process.recovered",
    subject="process-b",
    actor_id="system",
    detail="warm restart from disk",
)
await audit_b.log(
    event_type="task.resumed",
    subject="task-1",
    actor_id="agent://alice",
    detail="continuing pre-crash work",
)

valid, last = await audit_b.verify_chain()
print("continuous chain valid?", valid)
print("final seq             :", last)

continuous chain valid? True
final seq             : 5


Sanity check: a different key cannot verify the same chain. This is the guarantee — the chain authenticates the writer, not just its order.

In [30]:
wrong_seed = hashlib.sha256(b"WRONG-KEY-NOT-THE-ORIGINAL-32B--").digest()
wrong_key_audit = AuditLogger(recovery_store, signer=InProcessSigner(wrong_seed, ED25519))
# Note: do NOT call initialize() here — we want to verify from scratch.
valid, last = await wrong_key_audit.verify_chain()
print("wrong-key verify valid?", valid)
print("last verified seq      :", last)

Audit signature mismatch at seq 1


wrong-key verify valid? False
last verified seq      : 0


Equally important: a missing entry in the middle of the chain is detected by the sequence-gap check, not just a signature mismatch. Verification reports `False` and tells you the last seq it saw cleanly.

In [31]:
# Demonstrate the sequence-gap check on a fresh in-memory chain.
gap_backend = MemoryBackend()
gap_audit = AuditLogger(gap_backend, signer=InProcessSigner(RECOVERY_SEED, ED25519))
await gap_audit.initialize()
for i in range(5):
    await gap_audit.log(
        event_type=f"event_{i}",
        subject="x",
        actor_id="agent://a",
        detail=str(i),
    )
# Snip out the middle record — simulates a partial archive restore.
del gap_backend._streams["audit"]["audit"][2]
valid, last = await gap_audit.verify_chain()
print("gap-detected valid?", valid)
print("last good seq     :", last)

Audit sequence gap: expected 3, got 4


gap-detected valid? False
last good seq     : 2


Memory service recovery. Re-opening the same `TeamMemoryConfig.root` from a new process picks up the prior decisions log and any persisted entities. We don't have an entities flow set up here, but the decisions log is enough to demonstrate the pattern.

In [32]:
# Use the memory_root we set up in section 6 — same on-disk state.
svc_b = TeamMemoryService(config=TeamMemoryConfig(enabled=True, root=memory_root))
decisions_path = memory_root / "decisions.jsonl"
lines = decisions_path.read_text().splitlines() if decisions_path.exists() else []
print("decisions on disk after restart:", len(lines))
for line in lines:
    parsed = json.loads(line)
    print("  -", parsed.get("title", "<no title>"), "by", parsed["agent_id"])

# Append a post-restart decision; old decisions stay intact.
await svc_b.record_decision(
    decision={"title": "Verify chain on every boot"},
    agent_id="agent://carol",
)
post_lines = decisions_path.read_text().splitlines()
print("after appending: total lines =", len(post_lines))

decisions on disk after restart: 2
  - Adopt NatsBackend for shared storage by agent://alice
  - Rotate the audit signer quarterly by agent://bob
after appending: total lines = 3


## 9. Operational guidance — backups, durability, perms

Persistence is only as durable as the operational practices around it. A short checklist for production deployments:

**Backups.** For `NatsBackend`, durability and replication are JetStream's job — configure stream replicas and retention at the JetStream level (mirrors, source streams, or a periodic JetStream backup/restore) rather than trying to snapshot files yourself; there is no on-disk tree for this notebook's code to copy. `TeamFileStore`'s shared workspace *is* a plain directory tree, so back that up the conventional way (filesystem snapshot, `rsync`, etc.) — `shutil.copy2` already preserves mtimes for you on write.

**File permissions.** `TeamFileStore`'s shared directory and the files inside it should be locked down to the service user (`0o700` dir, `0o600` files) — the same hardening pattern used for `arctrust` trust-store files (`arctrust/02-keypairs-signing.ipynb`). JetStream-backed collections don't have local file permissions to set; access control there is a NATS auth/ACL concern.

**Signer / key rotation.** Rotate the `AuditLogger`'s `Signer` material periodically — for `InProcessSigner`, that means rotating the seed (loaded from your secret store, never hardcoded); for `VaultSigner`, it means rotating the key at the Vault Transit / HSM / KMS boundary. Either way, old chains stay verifiable under the old public key — `verify_chain` always checks against a *known* public key, so archive the retired key alongside the chain it signed. *Never* discard a key while its chain still needs to be verifiable.

**Retention.** JetStream streams have configurable retention (age, size, or interest-based) — set it deliberately per stream class (audit streams typically want long/permanent retention; ephemeral chat channels may not). This replaces "roll the JSONL file daily" — there is no local file to roll.

**Concurrent writers.** `NatsBackend` doesn't need a filesystem lock: every publish to a stream's subject is sequenced by the JetStream server itself, so concurrent writers across hosts and processes never collide on `seq` (demonstrated with `MemoryBackend`'s single-threaded equivalent in section 4). `MemoryBackend` is single-process by construction — never share one across processes; it isn't durable and was never meant to be.

In [33]:
# Demonstrate the file-permission hardening you'd apply in production —
# TeamFileStore is the real filesystem-backed component (NatsBackend has no
# local files to harden; that's a NATS auth/ACL concern instead).
import stat

perm_team_root = fresh_tmp("perms")
perm_store = TeamFileStore(team_root=perm_team_root)

perm_src = fresh_tmp("perms-src") / "confidential.txt"
perm_src.parent.mkdir(parents=True, exist_ok=True)
perm_src.write_text("sensitive report contents")

info = await perm_store.store(source_path=perm_src, agent_name="alice")
stored_file = Path(info["path"])

# After writing, harden the shared dir and the file.
os.chmod(perm_store.shared_dir, 0o700)
os.chmod(stored_file, 0o600)

print("shared_dir perms:", oct(stat.S_IMODE(perm_store.shared_dir.stat().st_mode)))
print("file perms      :", oct(stat.S_IMODE(stored_file.stat().st_mode)))

# Verify the API still works; only the OS-level access is restricted.
print("still readable  :", stored_file.read_text())

shared_dir perms: 0o700
file perms      : 0o600
still readable  : sensitive report contents


A second operational note: keep audit logs archived even after backend rotation. The `AuditRecord` model is a stable Pydantic schema, so a future investigator can re-parse decade-old JSONL with the same library.

In [34]:
# Show the AuditRecord schema is round-trippable — old logs stay readable.
raw = await recovery_store.read_stream("audit", "audit", after_seq=0, limit=2)
for r in raw:
    # Drop the transport-only "seq" mirror; the rest is plain Pydantic-validated data.
    rec_kwargs = {k: v for k, v in r.items() if k != "seq"}
    parsed = AuditRecord(**rec_kwargs)
    print(f"  seq={parsed.audit_seq}  {parsed.event_type}  by {parsed.actor_id}")

  seq=1  team.formed  by agent://alice
  seq=2  member.joined  by agent://alice


And finally: the same recovery flow applies if you swap backends. Take a `MemoryBackend` set up for a unit test, dump its records, and import them into another `StorageBackend` implementation — the data shape is identical because both implement the same Protocol. In production the destination would be a live `NatsBackend` (`await NatsBackend.connect(servers)`); we use a second `MemoryBackend` here since this notebook has no NATS server, but the loop below is exactly what a one-time migration script looks like either way.

In [35]:
# Cross-backend export/import — illustrates that the Protocol is the contract.
src_backend = MemoryBackend()
for i in range(3):
    await src_backend.write("entities", f"e{i}", {"id": f"e{i}", "n": i})

dst_backend = MemoryBackend()  # stand-in for a live NatsBackend in production
for key in await src_backend.list_keys("entities"):
    record = await src_backend.read("entities", key)
    if record is not None:
        await dst_backend.write("entities", key, record)

transferred = await dst_backend.list_keys("entities")
print("transferred keys:", transferred)

transferred keys: ['e0', 'e1', 'e2']


## 10. Summary

Persistence in ArcTeam is a federal-tier requirement implemented with a small, sharp set of components.

- **`StorageBackend` Protocol** is the contract every persistence client codes against — records (`read`/`write`/`delete`/`exists`/`query`/`list_keys`) and streams (`append`/`append_auto_seq`/`read_stream`/`read_last`/`get_stream_end_byte_pos`), plus durable consumers (`open_consumer`). Two implementations ship in the box: `MemoryBackend` (dependency-free, in-process, used everywhere in unit tests) and `NatsBackend` (production, backed by NATS JetStream). There is no file-based backend — durability comes from JetStream, not from a hand-rolled JSON/JSONL writer.
- **`NatsBackend`** maps records onto a JetStream KV bucket per collection and streams onto a JetStream stream per `(collection, key)`, with names hex-encoded into safe NATS tokens (`_bucket_name`, `_stream_name`, `_subject` — all pure, connection-free functions we exercised directly). The JetStream publish sequence is the authoritative `seq`; durable consumers give resume-after-restart with no client-side bookkeeping.
- **`TeamFileStore`** isolates per-agent shared workspaces under `<team_root>/shared/files/<agent>/`, validates with `_validate_path_within`, and avoids overwrites with numeric suffixes.
- **`TeamMemoryService` + `TeamMemoryConfig`** wire a knowledge-graph facade with a Null Object disabled mode, derive `entities_dir` and `index_path` from `root`, and append decisions to `<root>/decisions.jsonl` for cross-restart recall.
- **`AuditLogger`** chains a per-record Ed25519 (or ECDSA-P256) `Signer` signature across `AuditRecord` entries — each signature covers `prev_signature || canonical(record)` — calls `read_last` on `initialize()` to recover its position, and offers `verify_chain` to detect tampering, sequence gaps, and wrong-key verification attempts in batches.
- **Recovery flow** is: construct fresh service objects against the same durable store, `await audit.initialize()`, and continue. The chain spans process boundaries; old records remain Pydantic-parseable forever.
- **Operational guidance** — JetStream owns replication/retention for `NatsBackend`-held data; `TeamFileStore`'s plain directory tree gets conventional filesystem backup and mode `0o700`/`0o600`; rotate the `Signer`'s key material (not an HMAC secret) and archive retired public keys; concurrent writers never collide because JetStream sequences publishes server-side.

Public API exercised in this notebook: `StorageBackend`, `MemoryBackend`, `NatsBackend`, `TeamFileStore`, `TeamMemoryConfig`, `TeamMemoryService`, `AuditLogger`, `AuditRecord`, plus `InProcessSigner`/`build_signer`/`SignerConfig` from `arctrust.signer` and the `Classification`/`EntityMetadata` types from `arcteam.memory.types`.

**Next:** the arcteam track is now complete — `01-team-formation` (formation), `02-task-distribution` (work routing), `03-messaging-channels` (inter-agent comms), and this notebook (persistence) cover the full lifecycle. For the larger system, see `arcui/01-dashboard-bringup.ipynb` to surface team telemetry in a dashboard.